# 04 — Revenue scenarios and experiment design
Requires saved predictions from notebook 03; no refitting. All monetary quantities are in NTD. Historical 30-day-equivalent payments are proxies, not measured lost revenue or accounting MRR. Save rates are assumed, not identified from this dataset.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.json").exists() and (p / "src" / "common.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.common import paths, save_frame, load_frame
config, project_paths = paths(ROOT)
data_dir = project_paths["raw"]
processed_dir = project_paths["processed"]
figures_dir = project_paths["figures"]
tables_dir = project_paths["tables"]
artifacts_dir = project_paths["artifacts"]
cutoff = pd.Timestamp(config["cutoff"])


In [ ]:
from src.strategy import strategy_frame, select_policy
validation_predictions_saved = load_frame(artifacts_dir / 'validation_predictions.csv.gz')
test_predictions_saved = load_frame(artifacts_dir / 'test_predictions.csv.gz')
assert set(validation_predictions_saved.msno).isdisjoint(set(test_predictions_saved.msno))
strategy = strategy_frame(validation_predictions_saved)
print('Validation rows excluded for unknown value:', len(validation_predictions_saved)-len(strategy))


In [ ]:
capacity = int(np.ceil(len(strategy) * 0.10))
contact_fraction = capacity / len(strategy)

total_predicted_value = strategy["predicted_value_at_risk"].sum()
total_observed_value = strategy["observed_churn_associated_value"].sum()
total_churners = strategy["is_churn"].sum()


def summarize_targeting(selected):
    observed_value = selected["observed_churn_associated_value"].sum()

    return {
        "contacts": len(selected),
        "predicted_value_selected": selected["predicted_value_at_risk"].sum(),
        "observed_churn_value_selected": observed_value,
        "share_of_churn_value_captured": observed_value / total_observed_value,
        "share_of_churners_captured": selected["is_churn"].sum() / total_churners
    }


# Subscriber ID breaks any score ties reproducibly.
top_risk = strategy.sort_values(
    ["predicted_churn_probability", "msno"],
    ascending=[False, True]
).head(capacity)

top_value = strategy.sort_values(
    ["predicted_value_at_risk", "msno"],
    ascending=[False, True]
).head(capacity)

random_expected = {
    "contacts": capacity,
    "predicted_value_selected": contact_fraction * total_predicted_value,
    "observed_churn_value_selected": contact_fraction * total_observed_value,
    "share_of_churn_value_captured": contact_fraction,
    "share_of_churners_captured": contact_fraction
}

targeting_comparison = pd.DataFrame({
    "Random (expected)": random_expected,
    "Highest churn probability": summarize_targeting(top_risk),
    "Highest value at risk": summarize_targeting(top_value)
}).T

display(
    targeting_comparison.style.format({
        "contacts": "{:,.0f}",
        "predicted_value_selected": "{:,.2f}",
        "observed_churn_value_selected": "{:,.2f}",
        "share_of_churn_value_captured": "{:.2%}",
        "share_of_churners_captured": "{:.2%}"
    })
)

In [ ]:
save_rates = [0.05, 0.10, 0.20]
contact_costs = [1.0, 5.0, 10.0]

scenario_rows = []

for rule, row in targeting_comparison.iterrows():
    contacts = int(row["contacts"])
    predicted_value = row["predicted_value_selected"]

    for save_rate in save_rates:
        for contact_cost in contact_costs:
            campaign_cost = contacts * contact_cost
            revenue_protected = predicted_value * save_rate

            scenario_rows.append({
                "targeting_rule": rule,
                "save_rate": save_rate,
                "contact_cost": contact_cost,
                "campaign_cost": campaign_cost,
                "expected_revenue_protected": revenue_protected,
                "net_revenue_after_contact_cost": (
                    revenue_protected - campaign_cost
                ),
                "net_return_per_cost": (
                    (revenue_protected - campaign_cost) / campaign_cost
                ),
                "break_even_save_rate": campaign_cost / predicted_value
            })

scenarios = pd.DataFrame(scenario_rows)

# Display one illustrative scenario across all three targeting rules.
example = scenarios.loc[
    scenarios["save_rate"].eq(0.10)
    & scenarios["contact_cost"].eq(5.0)
]

display(
    example.style.format({
        "save_rate": "{:.0%}",
        "contact_cost": "{:.2f}",
        "campaign_cost": "{:,.2f}",
        "expected_revenue_protected": "{:,.2f}",
        "net_revenue_after_contact_cost": "{:,.2f}",
        "net_return_per_cost": "{:.1%}",
        "break_even_save_rate": "{:.2%}"
    })
)

In [ ]:
value_scenarios = scenarios.loc[
    scenarios["targeting_rule"].eq("Highest value at risk")
]

sensitivity = value_scenarios.pivot(
    index="save_rate",
    columns="contact_cost",
    values="net_revenue_after_contact_cost"
)

sensitivity.index = [
    f"{rate:.0%}" for rate in sensitivity.index
]
sensitivity.index.name = "Assumed save rate"
sensitivity.columns.name = "Cost per contact"

display(
    sensitivity.style
    .format("{:,.0f}")
    .background_gradient(cmap="RdYlGn", axis=None)
)

In [ ]:
assumed_save_rate = 0.10
cost_per_contact = 5.0

strategy["expected_net_per_contact"] = (
    assumed_save_rate * strategy["predicted_value_at_risk"]
    - cost_per_contact
)

economically_eligible = strategy.loc[
    strategy["expected_net_per_contact"].gt(0)
]

selected_campaign = (
    economically_eligible
    .sort_values(
        ["expected_net_per_contact", "msno"],
        ascending=[False, True]
    )
    .head(capacity)
)

expected_protected = (
    selected_campaign["predicted_value_at_risk"].sum()
    * assumed_save_rate
)
total_cost = len(selected_campaign) * cost_per_contact

print(f"Maximum contact capacity: {capacity:,}")
print(f"Subscribers with positive expected net: {len(economically_eligible):,}")
print(f"Selected contacts: {len(selected_campaign):,}")
print(f"Contact cost: {total_cost:,.2f}")
print(f"Expected revenue protected: {expected_protected:,.2f}")
print(f"Expected net after contact costs: {expected_protected - total_cost:,.2f}")

At an assumed 10% save rate and cost of 5 per contact, contact subscribers whose predicted 30-day-equivalent value at risk exceeds 50, ranked highest first, up to the capacity limit.

In [ ]:
test_strategy = strategy_frame(test_predictions_saved)
print('Test rows excluded for unknown value:', len(test_predictions_saved)-len(test_strategy))
test_capacity = int(np.ceil(len(test_strategy) * 0.10))

test_ranked = test_strategy.sort_values(
    ["predicted_value_at_risk", "msno"],
    ascending=[False, True]
)

test_full_capacity = test_ranked.head(test_capacity)

test_positive_benefit = test_ranked.loc[
    test_ranked["predicted_value_at_risk"] * 0.10 > 5.0
].head(test_capacity)

policy_rows = []

for name, selected in [
    ("Fill capacity", test_full_capacity),
    ("Positive benefit only", test_positive_benefit)
]:
    protected = selected["predicted_value_at_risk"].sum() * 0.10
    cost = len(selected) * 5.0

    policy_rows.append({
        "policy": name,
        "contacts": len(selected),
        "contact_cost": cost,
        "expected_revenue_protected": protected,
        "expected_net": protected - cost
    })

display(pd.DataFrame(policy_rows).round(2))

In [ ]:
test_strategy["observed_churn_value"] = (
    test_strategy["is_churn"]
    * test_strategy["revenue_30d_proxy"]
)

total_observed_value = test_strategy["observed_churn_value"].sum()
total_churners = test_strategy["is_churn"].sum()

checks = []

for name, selected in [
    ("Fill capacity", test_full_capacity),
    ("Positive benefit only", test_positive_benefit)
]:
    predicted_value = selected["predicted_value_at_risk"].sum()
    observed_value = (
        selected["is_churn"] * selected["revenue_30d_proxy"]
    ).sum()

    checks.append({
        "policy": name,
        "predicted_value_at_risk": predicted_value,
        "observed_churn_associated_value": observed_value,
        "prediction_error_relative_to_observed": (
            predicted_value / observed_value - 1
        ),
        "share_of_churn_value_captured": (
            observed_value / total_observed_value
        ),
        "share_of_churners_captured": (
            selected["is_churn"].sum() / total_churners
        )
    })

policy_checks = pd.DataFrame(checks)

display(
    policy_checks.style.format({
        "predicted_value_at_risk": "{:,.2f}",
        "observed_churn_associated_value": "{:,.2f}",
        "prediction_error_relative_to_observed": "{:+.2%}",
        "share_of_churn_value_captured": "{:.2%}",
        "share_of_churners_captured": "{:.2%}"
    })
)

A churn-based targeting policy selected 6.71% of test subscribers containing 55.32% of churn-associated 30-day-equivalent subscription value. Under an assumed 10% save rate and contact cost of 5, the policy projected 53,661 net revenue after contact costs—12.6% more than filling a 10% contact-capacity limit.

| Element | Design |
|---|---|
| Eligible population | Subscribers with upcoming expirations who pass the frozen eligibility and positive-benefit rules |
| Assignment | Randomly assign eligible subscribers 50/50 to outreach or business-as-usual control |
| Balance | Randomize within renewal-setting and predicted-risk groups |
| Initial intervention | One standardized, low-cost renewal reminder, without a discount |
| Primary outcome | Difference in churn rates, measured after each subscriber’s 30-day post-expiry renewal window |
| Economic outcome | Incremental revenue over a prespecified common measurement horizon, less incremental campaign costs |
| Guardrails | Unsubscribe rate, complaints, and support workload |
| Analysis | Compare everyone assigned to outreach with everyone assigned to control, regardless of whether they opened or received the message |

In [ ]:
from statistics import NormalDist
import math

# Estimated untreated churn rate in the proposed target population.
control_churn = selected_campaign["predicted_churn_probability"].mean()

alpha = 0.05
power = 0.80

normal = NormalDist()
z_alpha = normal.inv_cdf(1 - alpha / 2)
z_power = normal.inv_cdf(power)

sample_size_rows = []

for relative_reduction in [0.05, 0.10, 0.20]:
    treatment_churn = control_churn * (1 - relative_reduction)
    absolute_reduction = control_churn - treatment_churn
    pooled_rate = (control_churn + treatment_churn) / 2

    # Normal approximation for two independent proportions,
    # equal group sizes, and a two-sided test.
    n_per_arm = math.ceil(
        (
            z_alpha * math.sqrt(2 * pooled_rate * (1 - pooled_rate))
            + z_power * math.sqrt(
                control_churn * (1 - control_churn)
                + treatment_churn * (1 - treatment_churn)
            )
        ) ** 2
        / absolute_reduction ** 2
    )

    sample_size_rows.append({
        "relative_churn_reduction": relative_reduction,
        "control_churn": control_churn,
        "treatment_churn": treatment_churn,
        "absolute_reduction_pp": absolute_reduction * 100,
        "subscribers_per_arm": n_per_arm,
        "total_subscribers": 2 * n_per_arm,
        "outreach_cost_at_5": n_per_arm * 5
    })

experiment_sizes = pd.DataFrame(sample_size_rows)

display(
    experiment_sizes.style.format({
        "relative_churn_reduction": "{:.0%}",
        "control_churn": "{:.2%}",
        "treatment_churn": "{:.2%}",
        "absolute_reduction_pp": "{:.2f}",
        "subscribers_per_arm": "{:,.0f}",
        "total_subscribers": "{:,.0f}",
        "outreach_cost_at_5": "{:,.0f}"
    })
)

In [ ]:
targeting_comparison.to_csv(tables_dir / 'validation_targeting.csv')
scenarios.to_csv(tables_dir / 'scenario_grid.csv', index=False)
pd.DataFrame(policy_rows).to_csv(tables_dir / 'test_policy_economics.csv', index=False)
policy_checks.to_csv(tables_dir / 'test_policy_checks.csv', index=False)
experiment_sizes.to_csv(tables_dir / 'experiment_sizes.csv', index=False)
for label, frame in [('validation_selected', selected_campaign), ('test_selected', test_positive_benefit)]:
    save_frame(frame, artifacts_dir / f'{label}.csv.gz')
fig, ax = plt.subplots(figsize=(7, 5))
for cost, frame in value_scenarios.groupby('contact_cost'):
    frame = frame.sort_values('save_rate')
    ax.plot(frame.save_rate * 100, frame.net_return_per_cost * 100, marker='o', label=f'Contact cost {cost:g}')
ax.axhline(0, color='gray', linestyle='--')
ax.set(xlabel='Assumed relative churn reduction (%)', ylabel='Net revenue after contact cost / contact cost (%)', title='Fixed 10% value-ranked list: scenario returns')
ax.legend(); fig.tight_layout()
fig.savefig(figures_dir / 'scenario_returns.png', dpi=200)
plt.show()
print('Strategy tables and figures saved.')
